## Agentic Trading Bot - Complete Test Suite

This notebook provides comprehensive testing of all functionality in the agentic trading bot project, including:
- API health checks and endpoints
- Session management
- Document ingestion/upload
- Query processing and RAG capabilities
- Error handling and edge cases
- Agent workflows

**Last Updated**: 2026-05-13

# Section 1: Import Required Libraries

In [ ]:
import requests
import json
import asyncio
import uuid
from pathlib import Path
from typing import Dict, List, Any
import pandas as pd
import time
from datetime import datetime
import sys
from io import BytesIO
import shutil

# Add src to path for imports
sys.path.insert(0, str(Path.cwd().parent / "src"))

print("✓ All required libraries imported successfully")

# **Section 2: Load Project Modules**

✓ All required libraries imported successfully


# **Section 3: Set Up Configuration**

In [2]:
try:
    from infrastructure.config import get_config
    from infrastructure.logging import get_logger, setup_logging
    from infrastructure.container import get_app_context, get_container
    from domain.entities import QueryRequest, Message, MessageRole, ConversationContext
    from domain.exceptions import DomainException
    from presentation.dtos import QueryRequestDTO, IngestionResultDTO
    
    # Initialize logging
    setup_logging()
    logger = get_logger(__name__)
    
    print("✓ Project modules loaded successfully")
    print(f"✓ Logger initialized: {logger}")
except ImportError as e:
    print(f"⚠ Warning: Could not import some project modules: {e}")
    print("  This is expected if running from notebooks folder")
    logger = None

✓ Project modules loaded successfully
✓ Logger initialized: <infrastructure.logging.StructuredLogger object at 0x000001CE1E991480>


d:\AI_PROJECTS\agentic-trading-bot-project\.venv\lib\site-packages\pydantic\_internal\_config.py:386: UserWarning: Valid config keys have changed in V2:
* 'schema_extra' has been renamed to 'json_schema_extra'
  warnings.warn(message, UserWarning)


# **Section 4: Run Core Functionality**

In [3]:
# Configuration for testing
API_BASE_URL = "http://localhost:8000"
API_V1_PREFIX = "/api/v1"
FULL_API_URL = f"{API_BASE_URL}{API_V1_PREFIX}"

# Test session ID
TEST_SESSION_ID = str(uuid.uuid4())

# Test data
TEST_QUERY = "What is the current market trend for tech stocks?"
TEST_QUESTION = "Explain the recent trading patterns"

# Create a test data directory
test_data_dir = Path("test_data")
test_data_dir.mkdir(exist_ok=True)

# Create sample test file
sample_file_path = test_data_dir / "sample_document.txt"
sample_file_path.write_text("""
Trading Analysis Report - Q1 2026

Market Overview:
- Tech stocks up 15% YTD
- Federal Reserve maintains current interest rates
- Crypto market shows recovery

Key Findings:
1. AI companies leading market gains
2. Energy sector lagging behind
3. Real estate showing mixed signals

Recommendations:
- Diversify portfolio across sectors
- Consider defensive positions in tech
- Monitor macroeconomic indicators
""")

print("✓ Configuration setup complete")
print(f"  API Base URL: {FULL_API_URL}")
print(f"  Test Session ID: {TEST_SESSION_ID}")
print(f"  Test data directory: {test_data_dir.absolute()}")

✓ Configuration setup complete
  API Base URL: http://localhost:8000/api/v1
  Test Session ID: 80fb7a72-1e70-4561-a1f8-d9317917a5e8
  Test data directory: d:\AI_PROJECTS\agentic-trading-bot-project\notebooks\test_data


# **Section 5: Test Additional Features**

In [4]:
# Helper functions for testing
class APITester:
    """Helper class for API testing"""
    
    def __init__(self, base_url: str):
        self.base_url = base_url
        self.session_id = None
        self.test_results = []
    
    def log_result(self, test_name: str, passed: bool, details: str = ""):
        """Log test results"""
        status = "✓" if passed else "✗"
        self.test_results.append({
            "test": test_name,
            "passed": passed,
            "timestamp": datetime.now(),
            "details": details
        })
        print(f"{status} {test_name}: {details}")
    
    def health_check(self) -> Dict[str, Any]:
        """Test health endpoint"""
        try:
            response = requests.get(f"{self.base_url}/health", timeout=5)
            response.raise_for_status()
            data = response.json()
            self.log_result(
                "Health Check",
                response.status_code == 200,
                f"Status: {data.get('status')}, Version: {data.get('version')}"
            )
            return data
        except Exception as e:
            self.log_result("Health Check", False, f"Error: {str(e)}")
            return {"error": str(e)}
    
    def create_session(self) -> str:
        """Test session creation"""
        try:
            response = requests.post(f"{self.base_url}/session", timeout=5)
            response.raise_for_status()
            data = response.json()
            self.session_id = data.get("session_id")
            self.log_result(
                "Create Session",
                response.status_code == 200,
                f"Session ID: {self.session_id}"
            )
            return self.session_id
        except Exception as e:
            self.log_result("Create Session", False, f"Error: {str(e)}")
            return None
    
    def upload_documents(self, file_path: Path) -> Dict[str, Any]:
        """Test document upload"""
        try:
            if not file_path.exists():
                self.log_result("Upload Documents", False, f"File not found: {file_path}")
                return {"error": "File not found"}
            
            with open(file_path, 'rb') as f:
                files = {'files': (file_path.name, f)}
                params = {'session_id': self.session_id} if self.session_id else {}
                response = requests.post(
                    f"{self.base_url}/upload",
                    files=files,
                    params=params,
                    timeout=30
                )
            
            response.raise_for_status()
            data = response.json()
            self.log_result(
                "Upload Documents",
                response.status_code == 200,
                f"Documents: {data.get('documents_processed')}, Chunks: {data.get('chunks_created')}"
            )
            return data
        except Exception as e:
            self.log_result("Upload Documents", False, f"Error: {str(e)}")
            return {"error": str(e)}
    
    def query_chatbot(self, question: str) -> Dict[str, Any]:
        """Test query/chatbot functionality"""
        try:
            payload = {
                "question": question,
                "session_id": self.session_id or str(uuid.uuid4()),
                "context": "",
                "top_k": 3
            }
            
            response = requests.post(
                f"{self.base_url}/query",
                json=payload,
                timeout=30
            )
            
            response.raise_for_status()
            data = response.json()
            self.log_result(
                "Query Chatbot",
                response.status_code == 200,
                f"Confidence: {data.get('confidence')}, Sources: {len(data.get('sources', []))}"
            )
            return data
        except Exception as e:
            self.log_result("Query Chatbot", False, f"Error: {str(e)}")
            return {"error": str(e)}

# Initialize API Tester
print("Initializing API Tester...")
tester = APITester(FULL_API_URL)
print("✓ API Tester initialized")

Initializing API Tester...
✓ API Tester initialized


In [5]:
# Test 1: Health Check
print("\n" + "="*60)
print("TEST 1: HEALTH CHECK")
print("="*60)
health_response = tester.health_check()
print(f"Full Response: {json.dumps(health_response, indent=2)}")


TEST 1: HEALTH CHECK
✓ Health Check: Status: healthy, Version: 1.0.0
Full Response: {
  "status": "healthy",
  "version": "1.0.0"
}


In [6]:
# Test 2: Session Creation
print("\n" + "="*60)
print("TEST 2: SESSION CREATION")
print("="*60)
session_id = tester.create_session()
print(f"Session ID: {session_id}")


TEST 2: SESSION CREATION
✗ Create Session: Error: 500 Server Error: Internal Server Error for url: http://localhost:8000/api/v1/session
Session ID: None


In [ ]:
# Test 3: Document Upload/Ingestion
print("\n" + "="*60)
print("TEST 3: DOCUMENT UPLOAD/INGESTION")
print("="*60)
upload_response = tester.upload_documents(sample_file_path)
print(f"Full Response: {json.dumps(upload_response, indent=2)}")

In [ ]:
# Test 4: Query/Chatbot Functionality
print("\n" + "="*60)
print("TEST 4: QUERY/CHATBOT FUNCTIONALITY")
print("="*60)
query_response = tester.query_chatbot(TEST_QUESTION)
print(f"Full Response: {json.dumps(query_response, indent=2)}")

# **Section 6: Validate Outputs**

In [ ]:
# Test 5: Multiple Queries with Different Parameters
print("\n" + "="*60)
print("TEST 5: MULTIPLE QUERIES WITH DIFFERENT PARAMETERS")
print("="*60)

test_queries = [
    "What trading strategies are recommended?",
    "Tell me about market trends",
    "What sectors are performing well?",
    "Analyze the tech stock performance"
]

query_results = []
for query in test_queries:
    print(f"\nQuery: {query}")
    response = tester.query_chatbot(query)
    query_results.append({
        "query": query,
        "answer": response.get("answer", "N/A"),
        "confidence": response.get("confidence", 0),
        "num_sources": len(response.get("sources", [])),
        "execution_time": response.get("execution_time", 0)
    })
    print(f"  Confidence: {response.get('confidence')}")
    print(f"  Execution Time: {response.get('execution_time')}s")
    print(f"  Sources: {len(response.get('sources', []))}")

# **Section 7: Handle Edge Cases**

In [ ]:
# Validation functions
def validate_health_response(response: Dict) -> bool:
    """Validate health check response"""
    required_fields = ["status", "version"]
    return all(field in response for field in required_fields)

def validate_session_response(response: str) -> bool:
    """Validate session response"""
    try:
        uuid.UUID(response)
        return True
    except (ValueError, TypeError):
        return False

def validate_query_response(response: Dict) -> bool:
    """Validate query response"""
    required_fields = ["answer", "sources", "confidence", "execution_time"]
    has_all_fields = all(field in response for field in required_fields)
    
    # Additional validation
    if has_all_fields:
        confidence_valid = 0 <= response.get("confidence", 0) <= 1
        sources_valid = isinstance(response.get("sources"), list)
        return confidence_valid and sources_valid
    return False

# Run validations
print("\nValidating Health Response:")
health_valid = validate_health_response(health_response)
print(f"  Status: {'✓ VALID' if health_valid else '✗ INVALID'}")
assert health_valid, "Health response validation failed"

print("\nValidating Session Response:")
session_valid = validate_session_response(session_id)
print(f"  Status: {'✓ VALID' if session_valid else '✗ INVALID'}")
assert session_valid, "Session response validation failed"

print("\nValidating Query Responses:")
for idx, result in enumerate(query_results, 1):
    # Note: This validates the structure, even if answer might be empty
    print(f"  Query {idx}: Answer present={bool(result['answer'])}, "
          f"Confidence={result['confidence']:.2f}")

print("\n✓ All output validations completed")

# **Section 8: Visualize Results**

In [ ]:
# Test Edge Cases and Error Handling
print("\n" + "="*60)
print("TEST 7: EDGE CASES AND ERROR HANDLING")
print("="*60)

# Edge Case 1: Empty Query
print("\n1. Testing empty query:")
try:
    response = requests.post(
        f"{FULL_API_URL}/query",
        json={
            "question": "",
            "session_id": str(uuid.uuid4()),
            "context": "",
            "top_k": 3
        },
        timeout=10
    )
    print(f"  Status Code: {response.status_code}")
    print(f"  Response: {response.json()}")
except Exception as e:
    print(f"  Error: {e}")

# Edge Case 2: Invalid Session ID
print("\n2. Testing with invalid data types:")
try:
    response = requests.post(
        f"{FULL_API_URL}/query",
        json={
            "question": "Valid question?",
            "session_id": 12345,  # Invalid - should be string
            "context": "",
            "top_k": 3
        },
        timeout=10
    )
    print(f"  Status Code: {response.status_code}")
except Exception as e:
    print(f"  Error (Expected): {e}")

# Edge Case 3: Very Long Query
print("\n3. Testing with very long query:")
long_query = "What is the meaning of life? " * 100
try:
    response = tester.query_chatbot(long_query[:1000])  # Truncate to reasonable length
    print(f"  Status Code: 200")
    print(f"  Response received: {bool(response)}")
except Exception as e:
    print(f"  Error: {e}")

# Edge Case 4: Special Characters in Query
print("\n4. Testing with special characters:")
special_query = "What about $TSLA, $AAPL & crypto?"
try:
    response = tester.query_chatbot(special_query)
    print(f"  Response received: {bool(response)}")
except Exception as e:
    print(f"  Error: {e}")

# Edge Case 5: Concurrent Requests
print("\n5. Testing multiple concurrent queries:")
import concurrent.futures

def make_query(query_text):
    return tester.query_chatbot(query_text)

concurrent_queries = ["Market trend 1", "Market trend 2", "Market trend 3"]
try:
    with concurrent.futures.ThreadPoolExecutor(max_workers=3) as executor:
        futures = [executor.submit(make_query, q) for q in concurrent_queries]
        results = [f.result() for f in concurrent.futures.as_completed(futures)]
    print(f"  ✓ Successfully processed {len(results)} concurrent queries")
except Exception as e:
    print(f"  Error: {e}")

## Extra

# Cleanup and Summary
print("\n" + "="*60)
print("CLEANUP AND RECOMMENDATIONS")
print("="*60)

# Cleanup test files
import shutil
if test_data_dir.exists():
    try:
        shutil.rmtree(test_data_dir)
        print("✓ Test data directory cleaned up")
    except Exception as e:
        print(f"⚠ Could not clean up test directory: {e}")

print("\n📝 RECOMMENDATIONS FOR NEXT STEPS:")
print("""
1. **Health & Stability**
   - Ensure uvicorn server is running on localhost:8000
   - Monitor API response times under load
   - Check error logs for any issues

2. **Document Ingestion**
   - Test with various file formats (PDF, DOCX, etc.)
   - Validate chunk creation and retrieval
   - Verify metadata extraction

3. **Query Performance**
   - Test with larger document sets
   - Measure RAG accuracy metrics
   - Profile agent execution time

4. **Concurrent Operations**
   - Load test with multiple simultaneous queries
   - Monitor memory usage during ingestion
   - Test session management under stress

5. **Integration Testing**
   - Test agent workflows end-to-end
   - Verify trading strategy execution
   - Validate database persistence

6. **Production Readiness**
   - Run security vulnerability scan
   - Test error recovery mechanisms
   - Validate logging and monitoring
""")

print("\n✓ Test suite execution completed!")
print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

In [ ]:
# Query Performance Analysis
if query_results:
    print("\n" + "="*60)
    print("QUERY PERFORMANCE ANALYSIS")
    print("="*60)
    
    query_df = pd.DataFrame(query_results)
    print("\nQuery Results Summary:")
    print(query_df[['query', 'confidence', 'num_sources', 'execution_time']].to_string(index=False))
    
    print(f"\nAverage Confidence: {query_df['confidence'].mean():.3f}")
    print(f"Average Execution Time: {query_df['execution_time'].mean():.3f}s")
    print(f"Total Sources Retrieved: {query_df['num_sources'].sum()}")
else:
    print("\nNo query results to analyze")

In [ ]:
# Create a summary DataFrame of test results
test_summary_df = pd.DataFrame(tester.test_results)

print("\n" + "="*60)
print("TEST SUMMARY")
print("="*60)
print(test_summary_df.to_string(index=False))

# Calculate statistics
if test_summary_df.empty:
    print("\nNo test results to summarize")
else:
    passed_tests = test_summary_df['passed'].sum()
    total_tests = len(test_summary_df)
    pass_rate = (passed_tests / total_tests * 100) if total_tests > 0 else 0
    
    print(f"\nTotal Tests: {total_tests}")
    print(f"Passed: {passed_tests}")
    print(f"Failed: {total_tests - passed_tests}")
    print(f"Pass Rate: {pass_rate:.1f}%")